# || Feature Engineering : 'application_train' dataset ||

In [112]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
import warnings
print("All libraries imported!")
pd.set_option('display.float_format', lambda x: '%.3f' % x)
warnings.filterwarnings("ignore")

All libraries imported!


In [113]:
app_train=pd.read_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/raw/application_train.csv')
app_test=pd.read_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/raw/application_test.csv')
app_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.500,24700.500,...,0,0,0,0,0.000,0.000,0.000,0.000,0.000,1.000
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.500,35698.500,...,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.000,6750.000,...,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.500,29686.500,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.000,21865.500,...,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.000,254700.000,27558.000,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.000,269550.000,12001.500,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.000,677664.000,29979.000,...,0,0,0,0,1.000,0.000,0.000,1.000,0.000,1.000
307509,456254,1,Cash loans,F,N,Y,0,171000.000,370107.000,20205.000,...,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000


Before FE, we saw in the EDA that there was an unexpected value in the days employed that came out to be 1000 years. So, we will for now replace it with nan.

In [114]:
app_train.loc[app_train['DAYS_EMPLOYED']==365243,'DAYS_EMPLOYED']=np.nan
app_test.loc[app_test['DAYS_EMPLOYED']==365243,'DAYS_EMPLOYED']=np.nan

In [115]:
app_train['DAYS_EMPLOYED'].describe()

count   252137.000
mean     -2384.169
std       2338.360
min     -17912.000
25%      -3175.000
50%      -1648.000
75%       -767.000
max          0.000
Name: DAYS_EMPLOYED, dtype: float64

In [116]:
# converting days into years
app_train['age_years'] = abs(app_train['DAYS_BIRTH']) / 365
app_test['age_years'] = abs(app_test['DAYS_BIRTH']) / 365
app_train['employment_years'] = abs(app_train['DAYS_EMPLOYED']) / 365
app_test['employment_years'] = abs(app_test['DAYS_EMPLOYED']) / 365

### Features related to income, annuity, credit, days:

In [117]:
eps=1e-6

# EMI/NMI ratio
app_train['annuity_to_income']=app_train['AMT_ANNUITY']/(app_train['AMT_INCOME_TOTAL']+eps)
app_test['annuity_to_income']=app_test['AMT_ANNUITY']/(app_test['AMT_INCOME_TOTAL']+eps)

# credit to income ratio
app_train['credit_to_income']=app_train['AMT_CREDIT']/(app_train['AMT_INCOME_TOTAL']+eps)
app_test['credit_to_income']=app_test['AMT_CREDIT']/(app_test['AMT_INCOME_TOTAL']+eps)

# goods price to income ratio : weather he can afford the product or not
app_train['income_to_price']=app_train['AMT_INCOME_TOTAL']/(app_train['AMT_GOODS_PRICE']+eps)
app_test['income_to_price']=app_test['AMT_INCOME_TOTAL']/(app_test['AMT_GOODS_PRICE']+eps)

# income and days employed reltion
app_train['income_to_days']=app_train['AMT_INCOME_TOTAL']/((abs(app_train['DAYS_EMPLOYED'])/365)+eps)
app_test['income_to_days']=app_test['AMT_INCOME_TOTAL']/((abs(app_test['DAYS_EMPLOYED'])/365)+eps)

# income and age relation
app_train['income_to_age']=app_train['AMT_INCOME_TOTAL']/((abs(app_train['DAYS_BIRTH'])/365)+eps)
app_test['income_to_age']=app_test['AMT_INCOME_TOTAL']/((abs(app_test['DAYS_BIRTH'])/365)+eps)

# credit to annuity ratio
app_train['credit_to_annuity'] =app_train['AMT_CREDIT'] /(app_train['AMT_ANNUITY']+eps)
app_test['credit_to_annuity'] =app_test['AMT_CREDIT'] / (app_test['AMT_ANNUITY']+eps)

# goods to credit
app_train['goods_to_credit'] = app_train['AMT_GOODS_PRICE'] /(app_train['AMT_CREDIT']+eps)
app_test['goods_to_credit'] =app_test['AMT_GOODS_PRICE'] /(app_test['AMT_CREDIT']+eps)

# income divided into family members
app_train['income_per_person'] = app_train['AMT_INCOME_TOTAL']/(app_train['CNT_FAM_MEMBERS']+eps)
app_test['income_per_person'] = app_test['AMT_INCOME_TOTAL']/(app_test['CNT_FAM_MEMBERS']+eps)

# job to age ratio
app_train['employment_age_ratio'] = app_train['DAYS_EMPLOYED']/(app_train['DAYS_BIRTH']+eps)
app_test['employment_age_ratio'] = app_test['DAYS_EMPLOYED']/(app_test['DAYS_BIRTH']+eps)


### Family related features :

In [118]:
# child ratio in family
app_train['child_ratio'] = app_train['CNT_CHILDREN']/(app_train['CNT_FAM_MEMBERS'] + eps)
app_test['child_ratio'] = app_test['CNT_CHILDREN'] /(app_test['CNT_FAM_MEMBERS'] + eps)

# adult count in family
app_train['adult_count'] = app_train['CNT_FAM_MEMBERS'] -app_train['CNT_CHILDREN']
app_test['adult_count'] = app_test['CNT_FAM_MEMBERS'] -app_test['CNT_CHILDREN']

# credit per person
app_train['credit_per_person'] = app_train['AMT_CREDIT'] /(app_train['CNT_FAM_MEMBERS'] + eps)
app_test['credit_per_person'] = app_test['AMT_CREDIT'] /(app_test['CNT_FAM_MEMBERS'] + eps)

# annuity per person
app_train['annuity_per_person'] = app_train['AMT_ANNUITY'] /(app_train['CNT_FAM_MEMBERS'] + eps)
app_test['annuity_per_person'] = app_test['AMT_ANNUITY'] /(app_test['CNT_FAM_MEMBERS'] + eps)

### The external sources features

In [119]:
ext_cols=['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']

# mean
for df in [app_train,app_test]:
    df['ext_mean']=df[ext_cols].mean(axis=1)

# median
for df in [app_train,app_test]:
    df['ext_med']=df[ext_cols].median(axis=1)

# max
for df in [app_train,app_test]:
    df['ext_max']=df[ext_cols].max(axis=1)

# min
for df in [app_train,app_test]:
    df['ext_min']=df[ext_cols].min(axis=1)

# std
for df in [app_train,app_test]:
    df['ext_std']=df[ext_cols].std(axis=1)

# product
for df in [app_train,app_test]:
    df['ext_mul']=df['EXT_SOURCE_1']*df['EXT_SOURCE_2']*df['EXT_SOURCE_3']

# range
for df in [app_train,app_test]:
    df['ext_range']=df['ext_max']-df['ext_min']

# missing
for df in [app_train,app_test]:
    df['ext_missing']=df[ext_cols].isna().sum(axis=1)

# interactions
for df in [app_train, app_test]:
    df['ext_mean_x_age'] = (
        df['ext_mean']
        * (abs(df['DAYS_BIRTH']) / 365)
    )

for df in [app_train, app_test]:
    df['ext_mean_x_employment'] = (
        df['ext_mean']
        * (abs(df['DAYS_EMPLOYED']) / 365)
    )

for df in [app_train, app_test]:
    df['ext_mean_x_income'] = (
        df['ext_mean']
        * df['AMT_INCOME_TOTAL']
    )


### The documents provided flags :

In [120]:
doc_cols=[col for col in app_train.columns if 'FLAG_DOCUMENT_' in col]
for df in [app_test,app_train]:
    df['total_docs_provided']=df[doc_cols].sum(axis=1)

app_train['docs_absent']=len(doc_cols)-app_train['total_docs_provided']
app_test['docs_absent']=len(doc_cols)-app_test['total_docs_provided']

### Contact columns :

In [121]:
contact_cols = [
    'FLAG_MOBIL',
    'FLAG_EMP_PHONE',
    'FLAG_WORK_PHONE',
    'FLAG_CONT_MOBILE',
    'FLAG_PHONE',
    'FLAG_EMAIL'
]

for df in [app_train, app_test]:
    df['contact_count'] = df[contact_cols].sum(axis=1)

for df in [app_train, app_test]:
    df['missing_contact_count'] = (
        len(contact_cols) -
        df[contact_cols].sum(axis=1)
    )

### Address columns :

In [122]:
addr_cols = [
    'REG_REGION_NOT_LIVE_REGION',
    'REG_REGION_NOT_WORK_REGION',
    'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY',
    'REG_CITY_NOT_WORK_CITY',
    'LIVE_CITY_NOT_WORK_CITY'
]
# address mismatch
for df in [app_train, app_test]:
    df['address_mismatch_count'] = (
        df[addr_cols]
        .sum(axis=1)
    )

### Registration and ID:

In [123]:
for df in [app_train, app_test]:
    df['registration_age_ratio'] = (
        abs(df['DAYS_REGISTRATION'])
        /
        abs(df['DAYS_BIRTH'])
    )

for df in [app_train, app_test]:
    df['id_publish_age_ratio'] = (
        abs(df['DAYS_ID_PUBLISH'])
        /
        abs(df['DAYS_BIRTH'])
    )

### Housing misisng :

In [124]:
house_cols = [c for c in app_train.columns if '_AVG' in c or '_MODE' in c or '_MEDI' in c]

for df in [app_train, app_test]:
    df['housing_missing_count'] = df[house_cols].isnull().sum(axis=1)

# || Categorical Features ||

In [125]:
# binary encoding
for df in [app_train, app_test]:

    df['CODE_GENDER_BIN'] = (
        df['CODE_GENDER']
        .map({'M': 0, 'F': 1})
        .fillna(2)
        .astype('int8')
    )

    df['FLAG_OWN_CAR_BIN'] = (
        df['FLAG_OWN_CAR']
        .map({'N': 0, 'Y': 1})
        .astype('int8')
    )

    df['FLAG_OWN_REALTY_BIN'] = (
        df['FLAG_OWN_REALTY']
        .map({'N': 0, 'Y': 1})
        .astype('int8')
    )

In [126]:
# missing as category
cat_cols = app_train.select_dtypes(include='object').columns

for col in cat_cols:

    app_train[col] = app_train[col].fillna('Missing')
    app_test[col] = app_test[col].fillna('Missing')

In [127]:
freq_cols = [
    'ORGANIZATION_TYPE',
    'OCCUPATION_TYPE',
    'NAME_INCOME_TYPE'
]

for col in freq_cols:

    freq_map = (
        app_train[col]
        .value_counts(normalize=True)
        .to_dict()
    )

    app_train[f'{col}_FREQ'] = app_train[col].map(freq_map)

    app_test[f'{col}_FREQ'] = (
        app_test[col]
        .map(freq_map)
        .fillna(0)
    )

In [128]:
# organization is rare or not
org_freq = app_train['ORGANIZATION_TYPE'].value_counts(normalize=True)

rare_orgs = org_freq[org_freq < 0.01].index

for df in [app_train, app_test]:

    df['ORG_IS_RARE'] = (
        df['ORGANIZATION_TYPE']
        .isin(rare_orgs)
        .astype('int8')
    )

In [129]:
# self employed
for df in [app_train, app_test]:

    df['SELF_EMPLOYED_FLAG'] = (
        (df['NAME_INCOME_TYPE'] == 'Self-employed')
        .astype('int8')
    )

In [130]:
# employment segment
employment_map = {

    'Working': 'Working',
    'Commercial associate': 'Working',
    'State servant': 'Working',

    'Pensioner': 'Pensioner',

    'Student': 'Other',
    'Unemployed': 'Other',
    'Businessman': 'Other',
    'Maternity leave': 'Other'
}

for df in [app_train, app_test]:

    df['EMPLOYMENT_SEGMENT'] = (
        df['NAME_INCOME_TYPE']
        .map(employment_map)
        .fillna('Other')
    )

In [131]:
# skill group
high_skill = [
    'Managers',
    'Accountants',
    'Core staff',
    'HR staff',
    'IT staff',
    'Medicine staff',
    'High skill tech staff'
]

for df in [app_train, app_test]:

    df['OCCUPATION_SKILL_GROUP'] = np.where(
        df['OCCUPATION_TYPE'].isin(high_skill),
        'HighSkill',
        'Other'
    )

In [132]:
# categories for LGBM
lightgbm_cat_cols = [

    'NAME_CONTRACT_TYPE',
    'NAME_TYPE_SUITE',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
    'WEEKDAY_APPR_PROCESS_START',
    'ORGANIZATION_TYPE',
    'FONDKAPREMONT_MODE',
    'HOUSETYPE_MODE',
    'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE',

    'EMPLOYMENT_SEGMENT',
    'OCCUPATION_SKILL_GROUP'
]

for col in lightgbm_cat_cols:

    app_train[col] = app_train[col].astype('category')
    app_test[col] = app_test[col].astype('category')

In [133]:
len(app_train.columns)

166

In [134]:
# exporting app train and test dataframes
app_train.to_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/app_train_fe.parquet')
app_test.to_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/app_test_fe.parquet')